In [0]:

# DSAI3202 Cloud Computing - Phase 2
# Step 7: Final Model Evaluation
# Load the saved Random Forest and untouched test dataset

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# 1. Load the fixed Phase 2 test dataset
snapshot_root = (
    "abfss://curated@stflight60106354.dfs.core.windows.net/"
    "flight_delays/phase2/phase2_20251118_v1/"
)

test = spark.read.parquet(snapshot_root + "test/")

# 2. Load the trained Random Forest from MLflow
model_uri = "models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d"

saved_model = mlflow.sklearn.load_model(model_uri)

# 3. Use the threshold selected during validation
selected_threshold = 0.55

# 4. Define the same features used during training
feature_cols = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "CRSDepTime",
    "DepHour",
    "DayOfWeek",
    "IsWeekend",
    "Quarter"
]

target_col = "ArrDel15"

# 5. Check for target leakage and missing columns
assert "ArrDelay" not in feature_cols
assert target_col not in feature_cols

required_cols = feature_cols + [target_col]

assert all(
    column in test.columns
    for column in required_cols
), "Missing required columns"

# 6. Verify the test dataset
test_count = test.count()

assert test_count == 1127220, "Unexpected test row count"

print("FINAL HOLDOUT EVALUATION")
print("Total test flights:", test_count)
print("Model URI:", model_uri)
print("Selected threshold:", selected_threshold)
print("Features:", feature_cols)

print("PASS: Saved MLflow model loaded")
print("PASS: Test dataset loaded")
print("PASS: No target leakage detected")
print("PASS: Ready for final model evaluation")


FINAL HOLDOUT EVALUATION
Total test flights: 1127220
Model URI: models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d
Selected threshold: 0.55
Features: ['Reporting_Airline', 'Origin', 'Dest', 'Month', 'CRSDepTime', 'DepHour', 'DayOfWeek', 'IsWeekend', 'Quarter']
PASS: Saved MLflow model loaded
PASS: Test dataset loaded
PASS: No target leakage detected
PASS: Ready for final model evaluation


In [0]:

# Create a reproducible 5% sample of the final test data.
# The selection does not depend on the delay label.

hash_cols = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

test_sample = test.filter(
    F.pmod(
        F.xxhash64(*hash_cols),
        F.lit(20)
    ) == 0
)

sample_count = test_sample.count()

test_pd = test_sample.select(
    *feature_cols,
    target_col
).toPandas()

X_test = test_pd[feature_cols].copy()
y_test = test_pd[target_col].astype(int)

print("FINAL TEST SAMPLE")
print("Full test flights:", test_count)
print("Sampled flights:", sample_count)
print("Delay rate:", round(y_test.mean() * 100, 2), "%")

assert sample_count > 10000
assert len(X_test) == sample_count
assert test_pd[feature_cols].notna().all().all()
assert y_test.notna().all()

print("PASS: Reproducible test sample prepared")


FINAL TEST SAMPLE
Full test flights: 1127220
Sampled flights: 56901
Delay rate: 23.76 %
PASS: Reproducible test sample prepared


In [0]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Majority-class baseline established from training data.
baseline_pred = np.zeros(len(y_test), dtype=int)

# Saved Random Forest, with the fixed validation threshold.
rf_prob_test = saved_model.predict_proba(X_test)[:, 1]
rf_pred_test = (rf_prob_test >= selected_threshold).astype(int)

def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(actual, predicted), 4),
        "BalancedAccuracy": round(
            balanced_accuracy_score(actual, predicted), 4
        ),
        "Precision": round(
            precision_score(actual, predicted, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(actual, predicted, zero_division=0), 4
        ),
        "F1": round(
            f1_score(actual, predicted, zero_division=0), 4
        )
    }

final_results = pd.DataFrame([
    evaluate_model("Baseline", y_test, baseline_pred),
    evaluate_model("Random Forest", y_test, rf_pred_test)
])

print("FINAL HOLDOUT MODEL COMPARISON")
display(final_results)

print("Random Forest ROC-AUC:",
      round(roc_auc_score(y_test, rf_prob_test), 4))

print("\nRandom Forest confusion matrix:")
print(confusion_matrix(y_test, rf_pred_test))

print("\nRandom Forest classification report:")
print(classification_report(
    y_test,
    rf_pred_test,
    target_names=["On time", "Delayed"],
    zero_division=0
))

print("PASS: Both models evaluated on identical test records")


FINAL HOLDOUT MODEL COMPARISON


Model,Accuracy,BalancedAccuracy,Precision,Recall,F1
Baseline,0.7624,0.5,0.0,0.0,0.0
Random Forest,0.6042,0.5842,0.3106,0.546,0.396


Random Forest ROC-AUC: 0.615

Random Forest confusion matrix:
[[26995 16385]
 [ 6138  7383]]

Random Forest classification report:
              precision    recall  f1-score   support

     On time       0.81      0.62      0.71     43380
     Delayed       0.31      0.55      0.40     13521

    accuracy                           0.60     56901
   macro avg       0.56      0.58      0.55     56901
weighted avg       0.69      0.60      0.63     56901

PASS: Both models evaluated on identical test records


In [0]:

# Final test error analysis by airline.

errors = X_test.copy()

errors["Actual"] = y_test.to_numpy()
errors["Predicted"] = rf_pred_test

errors["FalsePositive"] = (
    (errors["Actual"] == 0) &
    (errors["Predicted"] == 1)
).astype(int)

errors["FalseNegative"] = (
    (errors["Actual"] == 1) &
    (errors["Predicted"] == 0)
).astype(int)

airline_errors = (
    errors.groupby("Reporting_Airline")
    .agg(
        Flights=("Actual", "size"),
        FalsePositives=("FalsePositive", "sum"),
        FalseNegatives=("FalseNegative", "sum")
    )
    .reset_index()
)

airline_errors = airline_errors[
    airline_errors["Flights"] >= 100
].copy()

airline_errors["ErrorRatePercent"] = (
    100 * (
        airline_errors["FalsePositives"] +
        airline_errors["FalseNegatives"]
    ) / airline_errors["Flights"]
).round(2)

airline_errors = airline_errors.sort_values(
    "ErrorRatePercent", ascending=False
)

print("FINAL HOLDOUT ERROR ANALYSIS")
display(airline_errors)

print("Total false positives:",
      int(errors["FalsePositive"].sum()))

print("Total false negatives:",
      int(errors["FalseNegative"].sum()))

print("PASS: Final error analysis completed")


FINAL HOLDOUT ERROR ANALYSIS


Reporting_Airline,Flights,FalsePositives,FalseNegatives,ErrorRatePercent
F9,1679,607,172,46.4
MQ,2411,827,243,44.38
YX,2998,992,335,44.26
AA,7971,2799,687,43.73
B6,1910,592,230,43.04
NK,1111,365,108,42.57
G4,1020,344,90,42.55
OH,1914,601,199,41.8
AS,1994,580,253,41.78
UA,6502,2069,621,41.37


Total false positives: 16385
Total false negatives: 6138
PASS: Final error analysis completed


In [0]:

from sklearn.metrics import f1_score, balanced_accuracy_score
import pandas as pd

# Final hypothesis evaluation on the fixed holdout sample.
# No model retraining or threshold changes.

baseline_macro_f1 = f1_score(
    y_test, baseline_pred, average="macro",
    zero_division=0
)

rf_macro_f1 = f1_score(
    y_test, rf_pred_test, average="macro",
    zero_division=0
)

baseline_balanced = balanced_accuracy_score(
    y_test, baseline_pred
)

rf_balanced = balanced_accuracy_score(
    y_test, rf_pred_test
)

hypothesis_results = pd.DataFrame([
    ("Baseline", baseline_macro_f1, baseline_balanced),
    ("Random Forest", rf_macro_f1, rf_balanced)
], columns=["Model", "MacroF1", "BalancedAccuracy"])

display(hypothesis_results)

print("Baseline macro-F1:", round(baseline_macro_f1, 4))
print("Random Forest macro-F1:", round(rf_macro_f1, 4))
print("Macro-F1 improvement:",
      round(rf_macro_f1 - baseline_macro_f1, 4))

if rf_macro_f1 > baseline_macro_f1:
    print("RESULT: Original macro-F1 hypothesis supported")
else:
    print("RESULT: Original macro-F1 hypothesis not supported")

print("PASS: Final hypothesis evaluation completed")


Model,MacroF1,BalancedAccuracy
Baseline,0.43258443772997873,0.5
Random Forest,0.5508099078585429,0.5841654363178512


Baseline macro-F1: 0.4326
Random Forest macro-F1: 0.5508
Macro-F1 improvement: 0.1182
RESULT: Original macro-F1 hypothesis supported
PASS: Final hypothesis evaluation completed
